[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/python_experiments_quickstart.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

# Phoenix 实验教程（Python）

本教程展示如何使用 Phoenix Experiments 系统评估和改进 AI Agent。你将学习创建数据集、定义在每个示例上运行 Agent 的任务函数，并使用代码评估器和 LLM 裁判衡量性能。最后，你可以运行实验比较不同 Agent 版本、跟踪持续改进，以数据支持开发和部署决策。

Notebook 分为四个主要部分，完整教程请参阅文档。

- **定义 Agent**：使用 `agno` 框架构建客服 Agent，提供工单分类和政策检索工具
- **创建数据集**：构建带真实标签的客服工单查询数据集，并上传到 Phoenix
- **定义实验**：创建任务函数与评估器，包括代码评估器和 LLM 裁判，运行实验衡量 Agent 性能并比较版本
- **通过实验迭代**：使用实验比较 Agent 版本，在部署前验证改进

# 安装依赖并设置密钥

In [ ]:
!pip install agno arize-phoenix openai openinference-instrumentation-agno openinference-instrumentation-openai

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
os.environ["PHOENIX_API_KEY"] = "your-phoenix-api-key"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "your-phoenix-collector-endpoint"

In [ ]:
from phoenix.otel import register

register(project_name="experiments-tutorial", auto_instrument=True)

# 定义客服 Agent

这个客服助手通过工单分类和检索相关政策帮助用户解决问题。它有两个工具：`classify_ticket` 将工单分为 billing、technical、account 或 other；`retrieve_policy` 根据工单类别获取适当的内部支持政策。

In [ ]:
from agno.models.openai import OpenAIChat
from agno.tools import tool
from openai import OpenAI

CATEGORIES = ["billing", "technical", "account", "other"]

openai_client = OpenAI()


@tool
def classify_ticket(ticket_text: str) -> str:
    """
    将客服工单分为：
    billing、technical、account 或 other。
    """

    messages = [
        {
            "role": "system",
            "content": (
                "将客服工单归入以下"
                "类别之一：billing、technical、account、other。"
                "只能回复类别名称。"
            ),
        },
        {
            "role": "user",
            "content": ticket_text,
        },
    ]

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    label = response.choices[0].message.content.strip().lower()

    if label not in CATEGORIES:
        return "other"

    return label

In [ ]:
POLICIES = {
    "billing": "Billing policy: Refunds are issued for duplicate charges within 7 days.",
    "technical": "Technical policy: Troubleshoot login issues, outages, and errors.",
    "account": "Account policy: Users can update email and password in account settings.",
    "other": "General support policy: Route to a human agent.",
}


@tool
def retrieve_policy(category: str) -> str:
    """检索内部支持政策。"""
    return POLICIES.get(category, POLICIES["other"])

In [ ]:
from agno.agent import Agent

support_agent = Agent(
    name="SupportAgent",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[classify_ticket, retrieve_policy],
    instructions="""
你是客服助手。

步骤：
1. 使用 classify_ticket 确定问题类别。
2. 使用 retrieve_policy 获取相关政策。
3. 根据政策写出有帮助、礼貌的回复。
不要编造政策。
""",
)

In [ ]:
sample_tickets = [
    "I was charged twice for my subscription this month.",
    "My app crashes every time I try to log in.",
    "How do I change the email on my account?",
    "This product is terrible and nothing works.",
]

for ticket in sample_tickets:
    support_agent.run(ticket)

# 第 1 节：创建数据集

In [ ]:
import pandas as pd
from phoenix.client import Client

data = [
    {
        "query": "I was charged twice for my subscription this month.",
        "expected_category": "billing",
    },
    {"query": "My app crashes every time I try to log in.", "expected_category": "technical"},
    {"query": "How do I change the email on my account?", "expected_category": "account"},
    {"query": "I want a refund because I was billed incorrectly.", "expected_category": "billing"},
    {"query": "The website shows a 500 error.", "expected_category": "technical"},
    {"query": "I forgot my password and cannot sign in.", "expected_category": "account"},
    {"query": "I was billed after canceling my subscription.", "expected_category": "billing"},
    {"query": "The app freezes on startup.", "expected_category": "technical"},
    {"query": "How can I update my billing address?", "expected_category": "account"},
    {"query": "Why was my credit card charged twice?", "expected_category": "billing"},
    {"query": "Push notifications are not working.", "expected_category": "technical"},
    {"query": "Can I change my username?", "expected_category": "account"},
    {"query": "I was charged even though my trial should be free.", "expected_category": "billing"},
    {"query": "The page won’t load on mobile.", "expected_category": "technical"},
    {"query": "How do I delete my account?", "expected_category": "account"},
    {
        "query": "I canceled last week but still see a pending charge and now the app won’t open.",
        "expected_category": "billing",
    },
    {
        "query": "Nothing works anymore and I don’t even know where to start.",
        "expected_category": "other",
    },
    {
        "query": "I updated my email and now I can’t log in — also was billed today.",
        "expected_category": "account",
    },
    {"query": "This service is unusable and I want my money back.", "expected_category": "billing"},
    {
        "query": "I think something is wrong with my account but support never responds.",
        "expected_category": "account",
    },
    {
        "query": "My subscription status looks wrong and the app crashes randomly.",
        "expected_category": "billing",
    },
    {
        "query": "Why am I being charged if I can’t access my account?",
        "expected_category": "billing",
    },
    {
        "query": "The app broke after the last update and now billing looks incorrect.",
        "expected_category": "technical",
    },
    {
        "query": "I’m locked out and still getting charged — please help.",
        "expected_category": "billing",
    },
    {
        "query": "This feels like both a billing and technical issue.",
        "expected_category": "billing",
    },
    {"query": "Everything worked yesterday, today nothing does.", "expected_category": "technical"},
    {
        "query": "I don’t recognize this charge and the app won’t load.",
        "expected_category": "billing",
    },
    {
        "query": "Account settings changed on their own and I was billed.",
        "expected_category": "account",
    },
    {"query": "I want to cancel but can’t log in.", "expected_category": "account"},
    {"query": "The system is broken and I’m losing money.", "expected_category": "billing"},
]

# 创建 DataFrame
dataset_df = pd.DataFrame(data)

# -----------------------------
# 上传数据集
# -----------------------------

px_client = Client()

dataset = px_client.datasets.create_dataset(
    dataframe=dataset_df,
    name="support-ticket-queries",
    input_keys=["query"],
    output_keys=["expected_category"],
)

# 第 2 节：定义实验

## 运行实验检查工具调用准确性（代码评估器）

这是上面的工具函数：

In [ ]:
def classify_ticket(ticket_text: str) -> str:
    """
    将客服工单分为：
    billing、technical、account 或 other。
    """

    messages = [
        {
            "role": "system",
            "content": (
                "将客服工单归入以下"
                "类别之一：billing、technical、account、other。"
                "只能回复类别名称。"
            ),
        },
        {
            "role": "user",
            "content": ticket_text,
        },
    ]

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    label = response.choices[0].message.content.strip().lower()

    if label not in CATEGORIES:
        return "other"

    return label

In [ ]:
def classify_ticket_task(input):
    """
    专门用于评估工具调用准确性的任务。
    """
    query = input.get("query")
    classification = classify_ticket(query)
    return classification

“基线”示例具有真实标签字段，因此可以使用代码评估器，检查任务输出是否符合预期。

In [ ]:
# 定义工具调用准确性的代码评估器
from phoenix.client.experiments import create_evaluator


@create_evaluator(kind="CODE", name="tool-call-accuracy")
def tool_call_accuracy(output: str, expected: dict) -> bool:
    """
    代码评估器，检查 classify_ticket 工具输出
    是否匹配数据集中的预期类别。
    """
    if expected is None:
        return None
    expected_category = expected.get("expected_category")
    return output.strip().lower() == expected_category.strip().lower()

In [ ]:
from phoenix.client.experiments import run_experiment

golden_dataset = px_client.datasets.get_dataset(dataset="support-ticket-queries")

experiment = run_experiment(
    dataset=golden_dataset,
    task=classify_ticket_task,
    evaluators=[tool_call_accuracy],
    experiment_name="tool call experiment",
    experiment_description="使用代码评估器，将 classify_ticket 工具输出与真实标签比较以评估准确性",
)

## 运行实验了解 Agent 整体表现（LLM 裁判评估器）

In [ ]:
def my_support_agent_task(input):
    """
    对数据集每行运行的任务函数。
    """
    query = input.get("query")

    # 使用查询调用 Agent
    response = support_agent.run(query)
    return response.content

In [ ]:
# 定义检查回复可执行性的 LLM 裁判评估器
from phoenix.client.resources.experiments.types import ExperimentEvaluation as EvaluationResult
from phoenix.evals import LLM, ClassificationEvaluator

# 定义提示词模板
support_response_actionability_judge = """
你正在评估客服 Agent 的回复。

判断回复是否可执行，是否帮助用户解决问题。

满足以下条件时标为 CORRECT：
- 直接回应用户的具体问题
- 提供具体步骤、指导或信息
- 清楚引导用户走向解决方案

以下情况标为 INCORRECT：
- 泛泛而谈、含糊或不具体
- 回避问题
- 没有明确后续步骤
- 仅用“联系支持”等说法推脱，却不提供指导

用户查询：
{input.query}

Agent 回复：
{output}

仅返回一个标签："correct" 或 "incorrect"。
"""

# 创建评估器
actionability_judge = ClassificationEvaluator(
    name="actionability-judge",
    prompt_template=support_response_actionability_judge,
    llm=LLM(model="gpt-5", provider="openai"),
    choices={"correct": 1.0, "incorrect": 0.0},
)


def call_actionability_judge(input, output):
    """
    可执行性裁判评估器的封装函数。
    run_experiment 接受函数而非评估器对象，因此需要此封装。
    """
    results = actionability_judge.evaluate({"input": input, "output": output})
    result = results[0]
    return EvaluationResult(score=result.score, label=result.label, explanation=result.explanation)

In [ ]:
from phoenix.client.experiments import run_experiment

experiment = run_experiment(
    dataset=dataset,
    task=my_support_agent_task,
    evaluators=[call_actionability_judge],
    experiment_name="support agent",
    experiment_description="使用可执行性裁判初步评估客服 Agent 回复的可执行性与帮助程度",
)

# 在 CI/CD 中使用实验

In [ ]:
# 改进 Agent，增强可执行性
# 此版本增强指令，以提高可执行性分数

improved_support_agent = Agent(
    name="SupportAgent",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[classify_ticket, retrieve_policy],
    instructions="""
你是客服助手。目标是提供具体、可执行的回复，直接帮助用户解决问题。

1. 使用 classify_ticket 确定问题类别。
2. 使用 retrieve_policy 获取相关政策。
3. 回复应：
   - 直接回应用户的具体问题
   - 包含检索到的政策信息
   - 提供用户可采取的清晰、具体后续步骤
   - 使用政策中的具体细节，例如“7 天内”而非“很快”
   - 避免“应该可以”“或许可以”等模糊措辞
   - 给出可执行指导，例如“前往设置 > 账号 > 邮箱”，而非“检查设置”

优秀回复示例：
"Based on your billing issue, here's what you can do: Refunds are issued for duplicate charges within 7 days. To request your refund, please [specific action]. You should see the refund processed within 7 business days."

较差回复示例：
"I understand your concern about billing. Please contact our support team for assistance with this matter."

不要编造政策。始终使用 retrieve_policy 返回的政策信息。
""",
)

In [ ]:
# 使用改进 Agent 的任务函数
def improved_support_agent_task(input):
    """
    使用增强可执行性指令的改进 Agent 的任务函数。
    """
    query = input.get("query")
    response = improved_support_agent.run(query)

    return response.content

In [ ]:
# 使用改进 Agent 运行实验，比较可执行性分数
from phoenix.client.experiments import run_experiment

# 获取数据集
improved_experiment = run_experiment(
    dataset=dataset,
    task=improved_support_agent_task,
    evaluators=[call_actionability_judge],
    experiment_name="改进的客服 Agent",
    experiment_description="增强指令以提高 Agent 可执行性，强调具体回复和明确后续步骤",
)